In [ ]:
!pip install -q scikit-learn xgboost lightgbm imbalanced-learn matplotlib seaborn pandas numpy

In [ ]:
# Notebook 3: Model Training & Evaluation

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import xgboost as xgb
import torch

from google.colab import drive
drive.mount('/content/drive')

OUT_DIR = Path('/content/drive/MyDrive/AIE_project/outputs')
ML_FEATURES_PATH = OUT_DIR / 'ml_features.csv'
MODEL_PATH = OUT_DIR / 'model.pkl'
PLOTS_DIR = OUT_DIR / 'plots'
PLOTS_DIR.mkdir(exist_ok=True)

df_features = pd.read_csv(ML_FEATURES_PATH)
print(f'Loaded features: {df_features.shape}')

Mounted at /content/drive
Loaded features: (1503227, 26)


In [ ]:
feature_cols = [c for c in df_features.columns if c not in ['priority', 'tweet_id']]
X = df_features[feature_cols].fillna(0)
y = df_features['priority']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train):,}, Test: {len(X_test):,}')

Train: 1,202,581, Test: 300,646


In [ ]:
USE_SMOTE = (y_train.value_counts(normalize=True).min() < 0.2)

def make_pipeline(model, use_smote=False):
    if use_smote:
        return ImbPipeline([('smote', SMOTE(random_state=42)),
                            ('scaler', StandardScaler()),
                            ('model', model)])
    else:
        return Pipeline([('scaler', StandardScaler()),
                         ('model', model)])

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced', n_jobs=-1, random_state=42),
    'XGBoost': xgb.XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=6,
                                 use_label_encoder=False, eval_metric='logloss',
                                 tree_method='hist', device='cuda' if torch.cuda.is_available() else 'cpu',
                                 random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=5, random_state=42),
}

pipelines = {name: make_pipeline(model, use_smote=USE_SMOTE) for name, model in models.items()}

In [ ]:
results = {}
for name, pipeline in pipelines.items():
    print(f'\nTraining {name}...')
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    results[name] = {
        'test_f1': f1_score(y_test, y_pred),
        'test_auc': roc_auc_score(y_test, y_prob),
        'pipeline': pipeline
    }
    print(f'  Test F1: {results[name]["test_f1"]:.4f}, AUC: {results[name]["test_auc"]:.4f}')

best_name = max(results, key=lambda k: results[k]['test_f1'])
best_pipeline = results[best_name]['pipeline']
print(f'\n✅ Best model: {best_name}')


Training Logistic Regression...
  Test F1: 0.4228, AUC: 0.9792

Training Random Forest...
  Test F1: 0.7116, AUC: 0.9955

Training XGBoost...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [13:29:40] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


  Test F1: 0.6258, AUC: 0.9960

Training Gradient Boosting...


In [ ]:
# Save model
with open(MODEL_PATH, 'wb') as f:
    pickle.dump({
        'pipeline': best_pipeline,
        'model_name': best_name,
        'feature_cols': feature_cols,
        'test_f1': results[best_name]['test_f1'],
        'test_auc': results[best_name]['test_auc'],
    }, f)

# Confusion matrix
y_pred_best = best_pipeline.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Urgent'])
disp.plot()
plt.title(f'Confusion Matrix – {best_name}')
plt.savefig(PLOTS_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

# Feature importance (permutation)
from sklearn.inspection import permutation_importance
X_test_sample = X_test.sample(min(20000, len(X_test)), random_state=42)
y_test_sample = y_test.loc[X_test_sample.index]
perm_imp = permutation_importance(best_pipeline, X_test_sample, y_test_sample,
                                  n_repeats=5, random_state=42, n_jobs=-1, scoring='f1')
imp_df = pd.DataFrame({'feature': feature_cols, 'importance': perm_imp.importances_mean}).sort_values('importance', ascending=False)
plt.figure(figsize=(10,6))
plt.barh(imp_df['feature'], imp_df['importance'], color='steelblue')
plt.title(f'Feature Importance – {best_name}')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'feature_importance.png', dpi=150)
plt.show()

print(f'Model and plots saved to {OUT_DIR}')